In [1]:
import numpy as np
import pandas as pd
import math

import xgboost as xgb
import lightgbm as lgb

import pickle
import tqdm

from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import accuracy_score

pd.set_option('display.max_columns',50)

In [2]:
detailed_regularM = pd.read_csv('./data/MRegularSeasonDetailedResults.csv')
print("Detailed regular data shape -->", detailed_regularM.shape)
detailed_regularW = pd.read_csv('./data/WRegularSeasonDetailedResults.csv')
print("Detailed regular data shape -->", detailed_regularW.shape)

Detailed regular data shape --> (118449, 34)
Detailed regular data shape --> (81308, 34)


In [3]:
def select_features(df):
    # generating and selecting important features
    df['WEffFGPerc'] = 100 * (df['WFGM'] + (0.5 * df['WFGM3']))/df['WFGA']
    df['LEffFGPerc'] = 100 * (df['LFGM'] + (0.5 * df['LFGM3']))/df['LFGA']
    df['WFTPerc'] = 100 * df['WFTM']/(df['WFTA']+1)
    df['LFTPerc'] = 100 * df['LFTM']/(df['LFTA']+1)
    df['WTOPerc'] = 100 * df['WTO'] / (df['WFGA'] - df['WOR'] + df['WTO'] + (0.44 * df['WFTA']))
    df['LTOPerc'] = 100 * df['LTO'] / (df['LFGA'] - df['LOR'] + df['LTO'] + (0.44 * df['LFTA']))
    df['WORPerc'] = 100 * df['WOR'] / (df['WOR'] + df['LDR'])
    df['LORPerc'] = 100 * df['LOR'] / (df['LOR'] + df['WDR'])
    df['WDefEff'] = (df['WStl'] + df['WBlk'])/ df['WPF']
    df['LDefEff'] = (df['LStl'] + df['LBlk'])/ df['LPF']

    drop_cols = ['WLoc','NumOT','WFGM','WFGA','LFGM','LFGA','WFGM3','WFGA3','WStl','WBlk','WPF','LFGM3','LFGA3','WFTM','WFTA','LStl','LBlk','LPF','LFTM','LFTA','WOR','WDR','LOR','LDR','WTO','LTO']
    df.drop(drop_cols, inplace=True, axis=1)

    return df

# replace Wins by Team A and B logic
def assign_teams(df):
    # true means winner goes to Team A
    np.random.seed(123)
    coin = np.random.rand(len(df)) < 0.5

    # team A stats based on coin toss
    teamA = pd.DataFrame({
        'Season': df['Season'],
        'DayNum': df['DayNum'],
        'TeamID' : np.where(coin, df['WTeamID'], df['LTeamID']),
        'Score' :  np.where(coin, df['WScore'], df['LScore']),
        'ORPerc': np.where(coin, df['WORPerc'], df['LORPerc']),
        'Ast': np.where(coin, df['WAst'], df['LAst']),
        'TOPerc': np.where(coin, df['WTOPerc'], df['LTOPerc']),
        'DefEff': np.where(coin, df['WDefEff'], df['LDefEff']),
        'EffFG': np.where(coin, df['WEffFGPerc'], df['LEffFGPerc']),
        'FTPerc': np.where(coin, df['WFTPerc'], df['LFTPerc']),
    })

    # team B stats based on coin toss
    teamB = pd.DataFrame({
        'TeamID' : np.where(coin, df['LTeamID'], df['WTeamID']),
        'Score' :  np.where(coin, df['LScore'], df['WScore']),
        'ORPerc': np.where(coin, df['LORPerc'], df['WORPerc']),
        'Ast': np.where(coin, df['LAst'], df['WAst']),
        'TOPerc': np.where(coin, df['LTOPerc'], df['WTOPerc']),
        'DefEff': np.where(coin, df['LDefEff'], df['WDefEff']),
        'EffFG': np.where(coin, df['LEffFGPerc'], df['WEffFGPerc']),
        'FTPerc': np.where(coin, df['LFTPerc'], df['WFTPerc']),
    })

    # Create Win column: 1 if Team A gets the winner's stats, 0 otherwise
    win = pd.Series(np.where(coin, 1, 0), name='Win')

    # Optionally, rename columns to clearly indicate Team A and B stats (except for Season)
    teamA = teamA.rename(columns=lambda x: x if x in ['Season','DayNum'] else x + '_A')
    teamB = teamB.rename(columns=lambda x: x if x in ['Season','DayNum'] else x + '_B')

    # Concatenate the two teams' stats and the Win column into one DataFrame
    new_df = pd.concat([teamA, teamB, win], axis=1)
    
    return new_df

feature_dfM = select_features(detailed_regularM)
final_dfM = assign_teams(feature_dfM)
display(final_dfM.head(10))

feature_dfW = select_features(detailed_regularW)
final_dfW = assign_teams(feature_dfW)
display(final_dfW.head(10))

,Season,DayNum,TeamID_A,Score_A,ORPerc_A,Ast_A,TOPerc_A,DefEff_A,EffFG_A,FTPerc_A,TeamID_B,Score_B,ORPerc_B,Ast_B,TOPerc_B,DefEff_B,EffFG_B,FTPerc_B,Win
0,2003,10,1328,62,29.411765,8,25.466893,0.550000,43.396226,69.565217,1104,68,38.888889,13,30.699413,0.363636,49.137931,57.894737,0
1,2003,10,1272,70,37.500000,16,19.016969,0.444444,48.387097,50.000000,1393,63,41.666667,7,17.699115,0.875000,40.298507,42.857143,1
2,2003,11,1266,73,43.589744,15,15.683814,0.280000,48.275862,56.666667,1437,61,54.385965,9,18.714910,0.304348,32.191781,58.333333,1
3,2003,11,1457,50,47.222222,9,32.986111,0.304348,42.857143,50.000000,1296,56,23.076923,11,20.818876,0.888889,51.315789,53.125000,0
4,2003,11,1208,71,48.837209,12,15.903308,0.571429,43.548387,60.714286,1400,77,53.125000,12,21.971124,0.400000,54.098361,78.571429,0
5,2003,11,1458,81,35.294118,12,13.661202,0.666667,50.877193,82.142857,1186,55,20.000000,8,28.580024,0.280000,46.739130,66.666667,1
6,2003,12,1236,62,33.333333,11,40.365985,0.500000,51.219512,68.965517,1161,80,38.235294,14,22.321429,0.480000,43.636364,80.000000,0
7,2003,12,1457,61,18.604651,10,19.705728,1.222222,37.288136,70.833333,1186,75,34.210526,19,24.598654,0.428571,48.387097,68.181818,0
8,2003,12,1194,71,25.714286,9,22.997835,0.478261,52.586207,52.631579,1156,66,37.142857,13,32.946758,0.555556,51.923077,42.857143,1
9,2003,12,1458,84,37.837838,11,8.907363,0.923077,51.492537,75.000000,1296,56,29.032258,10,27.157514,0.222222,47.115385,53.846154,1


,Season,DayNum,TeamID_A,Score_A,ORPerc_A,Ast_A,TOPerc_A,DefEff_A,EffFG_A,FTPerc_A,TeamID_B,Score_B,ORPerc_B,Ast_B,TOPerc_B,DefEff_B,EffFG_B,FTPerc_B,Win
0,2010,11,3237,49,29.729730,11,32.670455,0.684211,39.814815,54.545455,3103,63,27.027027,14,25.582717,0.466667,47.222222,60.000000,0
1,2010,11,3104,73,38.095238,15,25.536261,0.280000,45.967742,55.172414,3399,68,31.111111,7,24.727992,0.222222,42.857143,50.000000,1
2,2010,11,3110,71,37.837838,18,19.613760,0.470588,51.612903,53.846154,3224,59,42.500000,8,22.686025,0.400000,34.482759,79.166667,1
3,2010,11,3267,58,35.483871,15,14.864865,1.357143,28.378378,61.538462,3111,63,21.428571,14,35.083160,0.833333,55.769231,50.000000,0
4,2010,11,3447,70,38.888889,12,18.363064,0.428571,39.864865,50.000000,3119,74,30.434783,18,14.504219,0.444444,45.270270,58.333333,0
5,2010,11,3120,70,42.857143,13,21.097046,0.423077,36.153846,62.162162,3407,65,39.024390,13,36.359077,0.357143,46.296296,50.000000,1
6,2010,11,3146,76,33.333333,10,19.893899,0.454545,43.333333,66.666667,3125,85,39.393939,15,27.047913,0.259259,53.571429,78.125000,0
7,2010,11,3152,41,32.432432,7,39.149888,0.529412,33.653846,66.666667,3132,76,31.250000,19,26.330377,1.312500,55.263158,86.666667,0
8,2010,11,3138,87,42.105263,16,32.213530,1.200000,54.464286,59.090909,3310,79,41.379310,19,23.820327,0.428571,42.352941,46.666667,1
9,2010,11,3140,73,52.000000,21,20.673361,0.640000,45.270270,42.857143,3430,51,28.571429,4,33.783784,0.200000,37.804878,83.333333,1


In [4]:
def compute_rolling_stats(df):
    # -------------------------------------------
    # STEP 1: Compute Differential Columns for Each Matchup
    # -------------------------------------------
    # For Team A perspective:
    df['Score_diff_A']   = df['Score_A']   - df['Score_B']
    df['ORPerc_diff_A']  = df['ORPerc_A']  - df['ORPerc_B']
    df['Ast_diff_A']     = df['Ast_A']     - df['Ast_B']
    df['TOPerc_diff_A']  = df['TOPerc_A']  - df['TOPerc_B']
    df['DefEff_diff_A']  = df['DefEff_A']  - df['DefEff_B']
    df['EffFG_diff_A']   = df['EffFG_A']   - df['EffFG_B']
    df['FTPerc_diff_A']  = df['FTPerc_A']  - df['FTPerc_B']
    
    # For Team B perspective (reverse the differential):
    df['Score_diff_B']   = df['Score_B']   - df['Score_A']
    df['ORPerc_diff_B']  = df['ORPerc_B']  - df['ORPerc_A']
    df['Ast_diff_B']     = df['Ast_B']     - df['Ast_A']
    df['TOPerc_diff_B']  = df['TOPerc_B']  - df['TOPerc_A']
    df['DefEff_diff_B']  = df['DefEff_B']  - df['DefEff_A']
    df['EffFG_diff_B']   = df['EffFG_B']   - df['EffFG_A']
    df['FTPerc_diff_B']  = df['FTPerc_B']  - df['FTPerc_A']
    
    # Create the regression target: Point Margin = Score_A - Score_B.
    df['Point_Margin'] = df['Score_A'] - df['Score_B']
    
    # -------------------------------------------
    # STEP 2: Convert Wide Data to Long Format
    # -------------------------------------------
    # For rolling computations it is easier if each row represents a single team's performance.
    
    # Create a DataFrame for Team A rows:
    df_A = df[['Season', 'DayNum', 'TeamID_A', 
               'Score_diff_A', 'ORPerc_diff_A', 'Ast_diff_A', 'TOPerc_diff_A',
               'DefEff_diff_A', 'EffFG_diff_A', 'FTPerc_diff_A']].copy()
    df_A.rename(columns={
        'TeamID_A': 'Team',
        'Score_diff_A': 'Score_diff',
        'ORPerc_diff_A': 'ORPerc_diff',
        'Ast_diff_A': 'Ast_diff',
        'TOPerc_diff_A': 'TOPerc_diff',
        'DefEff_diff_A': 'DefEff_diff',
        'EffFG_diff_A': 'EffFG_diff',
        'FTPerc_diff_A': 'FTPerc_diff'
    }, inplace=True)
    
    # Create a DataFrame for Team B rows:
    df_B = df[['Season', 'DayNum', 'TeamID_B', 
               'Score_diff_B', 'ORPerc_diff_B', 'Ast_diff_B', 'TOPerc_diff_B',
               'DefEff_diff_B', 'EffFG_diff_B', 'FTPerc_diff_B']].copy()
    df_B.rename(columns={
        'TeamID_B': 'Team',
        'Score_diff_B': 'Score_diff',
        'ORPerc_diff_B': 'ORPerc_diff',
        'Ast_diff_B': 'Ast_diff',
        'TOPerc_diff_B': 'TOPerc_diff',
        'DefEff_diff_B': 'DefEff_diff',
        'EffFG_diff_B': 'EffFG_diff',
        'FTPerc_diff_B': 'FTPerc_diff'
    }, inplace=True)
    
    # Combine Team A and Team B rows into one long DataFrame:
    df_long = pd.concat([df_A, df_B], ignore_index=True)
    
    # -------------------------------------------
    # STEP 3: Compute 7-Game Rolling Averages for Differential Stats
    # -------------------------------------------
    # Sort by Season, Team, and DayNum so that each team's games are in chronological order.
    df_long.sort_values(by=['Season', 'Team', 'DayNum'], inplace=True)
    
    # Define a function to compute rolling averages using a 7-game window.
    def compute_rolling(group, window=10):
        group = group.copy()
        group['Score_diff_7']  = group['Score_diff'].rolling(window=window, min_periods=window).mean()
        group['ORPerc_diff_7'] = group['ORPerc_diff'].rolling(window=window, min_periods=window).mean()
        group['Ast_diff_7']    = group['Ast_diff'].rolling(window=window, min_periods=window).mean()
        group['TOPerc_diff_7'] = group['TOPerc_diff'].rolling(window=window, min_periods=window).mean()
        group['DefEff_diff_7'] = group['DefEff_diff'].rolling(window=window, min_periods=window).mean()
        group['EffFG_diff_7']  = group['EffFG_diff'].rolling(window=window, min_periods=window).mean()
        group['FTPerc_diff_7'] = group['FTPerc_diff'].rolling(window=window, min_periods=window).mean()
        return group
    
    df_long = df_long.groupby(['Season', 'Team']).apply(compute_rolling, window=7)
    df_long.reset_index(drop=True, inplace=True)
    
    # -------------------------------------------
    # STEP 4: Merge Rolling Stats Back into the Original (Wide) Matchup-Level DataFrame
    # -------------------------------------------
    # We need to merge the rolling stats for Team A and Team B back into the original DataFrame.
    
    # Specify the columns from the long DataFrame that contain the rolling averages:
    rolling_cols = ['Season', 'DayNum', 'Team', 
                    'Score_diff_7', 'ORPerc_diff_7', 'Ast_diff_7', 'TOPerc_diff_7',
                    'DefEff_diff_7', 'EffFG_diff_7', 'FTPerc_diff_7']
    
    # Extract rolling stats for Team A:
    rolling_A = df_long[rolling_cols].copy()
    df = df.merge(rolling_A, left_on=['Season', 'DayNum', 'TeamID_A'],
                  right_on=['Season', 'DayNum', 'Team'], how='left', suffixes=('', '_A'))
    df.rename(columns={
        'Score_diff_7': 'Score_diff_7_A',
        'ORPerc_diff_7': 'ORPerc_diff_7_A',
        'Ast_diff_7': 'Ast_diff_7_A',
        'TOPerc_diff_7': 'TOPerc_diff_7_A',
        'DefEff_diff_7': 'DefEff_diff_7_A',
        'EffFG_diff_7': 'EffFG_diff_7_A',
        'FTPerc_diff_7': 'FTPerc_diff_7_A'
    }, inplace=True)
    df.drop('Team', axis=1, inplace=True)
    
    # Extract rolling stats for Team B:
    rolling_B = df_long[rolling_cols].copy()
    df = df.merge(rolling_B, left_on=['Season', 'DayNum', 'TeamID_B'],
                  right_on=['Season', 'DayNum', 'Team'], how='left', suffixes=('', '_B'))
    df.rename(columns={
        'Score_diff_7': 'Score_diff_7_B',
        'ORPerc_diff_7': 'ORPerc_diff_7_B',
        'Ast_diff_7': 'Ast_diff_7_B',
        'TOPerc_diff_7': 'TOPerc_diff_7_B',
        'DefEff_diff_7': 'DefEff_diff_7_B',
        'EffFG_diff_7': 'EffFG_diff_7_B',
        'FTPerc_diff_7': 'FTPerc_diff_7_B'
    }, inplace=True)
    df.drop('Team', axis=1, inplace=True)
    
    # -------------------------------------------
    # STEP 5: Create Final DataFrame and Drop Original Columns
    # -------------------------------------------
    # We keep only the keys, the rolling stats, and target variables.
    # removed Score_diff, Assists after checking correlation
    final_cols = ['Season', 'DayNum', 'TeamID_A', 'TeamID_B',
                  'ORPerc_diff_7_A', 'TOPerc_diff_7_A',
                  'DefEff_diff_7_A', 'EffFG_diff_7_A', 'FTPerc_diff_7_A',
                  'ORPerc_diff_7_B', 'TOPerc_diff_7_B',
                  'DefEff_diff_7_B', 'EffFG_diff_7_B', 'FTPerc_diff_7_B',
                  'Win', 'Point_Margin']
    
    df_final = df[final_cols].dropna().reset_index(drop=True)

    return (df_long[rolling_cols].copy(), df_final)

df_longM, final_dfM = compute_rolling_stats(final_dfM)
df_longW, final_dfW = compute_rolling_stats(final_dfW)

C:\Users\siddh\AppData\Local\Temp\ipykernel_25052\2507245090.py:82: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_long = df_long.groupby(['Season', 'Team']).apply(compute_rolling, window=7)
C:\Users\siddh\AppData\Local\Temp\ipykernel_25052\2507245090.py:82: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_long = df_long.groupby(['Season', 'Team']).apply(compute_rolling, window=7)


In [5]:
df_longM.tail()

,Season,DayNum,Team,Score_diff_7,ORPerc_diff_7,Ast_diff_7,TOPerc_diff_7,DefEff_diff_7,EffFG_diff_7,FTPerc_diff_7
236893,2025,103,1480,-6.714286,1.397058,-2.857143,-1.031445,0.174720,-7.891073,9.041144
236894,2025,106,1480,-12.142857,-0.773671,-4.000000,1.197422,-0.026187,-9.717385,2.350058
236895,2025,108,1480,-13.000000,-2.097104,-2.428571,2.082404,-0.066984,-8.437088,2.041668
236896,2025,112,1480,-12.428571,-2.501287,-1.142857,3.050552,-0.143247,-6.105439,-3.220364
236897,2025,114,1480,-10.714286,-0.426419,-1.142857,2.441679,-0.017003,-5.561034,-6.629455


In [6]:
final_dfM.head()

,Season,DayNum,TeamID_A,TeamID_B,ORPerc_diff_7_A,TOPerc_diff_7_A,DefEff_diff_7_A,EffFG_diff_7_A,FTPerc_diff_7_A,ORPerc_diff_7_B,TOPerc_diff_7_B,DefEff_diff_7_B,EffFG_diff_7_B,FTPerc_diff_7_B,Win,Point_Margin
0,2003,33,1232,1183,-0.951985,-2.758769,0.120962,-4.983341,-12.428746,-2.439401,-2.613924,-0.132108,-7.448693,7.649894,1,6
1,2003,33,1237,1292,0.430207,2.448383,-0.130675,-15.615107,0.690647,0.975224,5.027691,-0.311933,-0.551846,-6.643701,0,-4
2,2003,35,1231,1435,0.979515,-0.298213,0.387308,8.557617,11.411400,0.485802,-1.763459,-0.014092,13.656050,-6.894016,1,17
3,2003,35,1250,1162,1.180624,3.828888,0.077071,1.456485,11.409258,-6.304264,1.474739,-0.283262,-11.607017,-10.361916,1,16
4,2003,37,1140,1360,0.619789,1.986108,0.035920,13.024963,10.488572,10.422564,6.703842,-0.103381,0.635117,-5.081380,1,15


In [7]:
train_dataM = final_dfM[final_dfM['Season'].isin(range(2003, 2025))].reset_index(drop=True)
train_dataM.to_csv('./data/final_train_dataM.csv', index=False)
train_dataW = final_dfW[final_dfW['Season'].isin(range(2003, 2025))].reset_index(drop=True)
train_dataW.to_csv('./data/final_train_dataW.csv', index=False)

In [8]:
# Define your features and target column
feature_cols = [
    'ORPerc_diff_7_A', 'TOPerc_diff_7_A', 'DefEff_diff_7_A',
    'EffFG_diff_7_A', 'FTPerc_diff_7_A', 'ORPerc_diff_7_B',
    'TOPerc_diff_7_B', 'DefEff_diff_7_B', 'EffFG_diff_7_B',
    'FTPerc_diff_7_B'
]
target_col = 'Point_Margin'

def normal_cdf(x):
    # Vectorize math.erf so it can handle array inputs
    erf_vectorized = np.vectorize(lambda t: math.erf(t))
    return 0.5 * (1 + erf_vectorized(x / np.sqrt(2)))

def margin_to_probability(margin, scale=10.0):
    """
    Convert predicted margin (scalar or array) to win probability for Team A
    using a Normal CDF approximation.
    """
    return normal_cdf(margin / scale)

# Define the Brier score function
def brier_score(y_true, y_prob):
    return np.mean((y_true - y_prob)**2)

In [9]:
# Define your feature columns and target column
feature_cols = [
    'ORPerc_diff_7_A', 'TOPerc_diff_7_A', 'DefEff_diff_7_A',
    'EffFG_diff_7_A', 'FTPerc_diff_7_A', 'ORPerc_diff_7_B',
    'TOPerc_diff_7_B', 'DefEff_diff_7_B', 'EffFG_diff_7_B',
    'FTPerc_diff_7_B'
]
target_col = 'Point_Margin'

XM = train_dataM[feature_cols]
yM = train_dataM[target_col].values

# Set up K-fold cross validation
n_splits = 10
kfM = KFold(n_splits=n_splits, shuffle=True, random_state=123)

brier_scores = []
accuracy_scores = []

for fold, (train_index, val_index) in enumerate(kfM.split(XM), 1):
    # Create training and validation splits for the fold
    X_train_cv, X_val_cv = XM.iloc[train_index], XM.iloc[val_index]
    y_train_cv, y_val_cv = yM[train_index], yM[val_index]
    
    # Initialize the LightGBM regressor with your fixed parameters
    modelM = lgb.LGBMRegressor(
        n_estimators=500,
        learning_rate=0.01,
        random_state=123,
        verbose=-1
    )
    
    # Train the model
    modelM.fit(X_train_cv, y_train_cv)
    
    # Predict the continuous point margin on the validation fold
    y_val_pred_margin = modelM.predict(X_val_cv)
    
    # Convert predicted margins to win probabilities
    y_val_pred_prob = margin_to_probability(y_val_pred_margin)
    
    # Convert true margins to binary outcomes (win=1 if margin > 0, else 0)
    y_val_true_label = (y_val_cv > 0).astype(int)
    
    # For accuracy, threshold the predicted probabilities at 0.5
    y_val_pred_label = (y_val_pred_prob > 0.5).astype(int)
    
    # Compute Brier score for the fold
    fold_brier = brier_score(y_val_true_label, y_val_pred_prob)
    brier_scores.append(fold_brier)
    
    # Compute accuracy score for the fold
    fold_accuracy = accuracy_score(y_val_true_label, y_val_pred_label)
    accuracy_scores.append(fold_accuracy)
    
    print(f"Fold {fold} - Brier Score: {fold_brier:.4f}, Accuracy: {fold_accuracy:.4f}")

# Compute average scores across folds
avg_brier = np.mean(brier_scores)
avg_accuracy = np.mean(accuracy_scores)
print(f"\nAverage Brier Score over {n_splits} folds: {avg_brier:.4f}")
print(f"Average Accuracy over {n_splits} folds: {avg_accuracy:.4f}")

Fold 1 - Brier Score: 0.1606, Accuracy: 0.7639
Fold 2 - Brier Score: 0.1644, Accuracy: 0.7556
Fold 3 - Brier Score: 0.1607, Accuracy: 0.7600
Fold 4 - Brier Score: 0.1661, Accuracy: 0.7524
Fold 5 - Brier Score: 0.1664, Accuracy: 0.7520
Fold 6 - Brier Score: 0.1615, Accuracy: 0.7615
Fold 7 - Brier Score: 0.1633, Accuracy: 0.7548
Fold 8 - Brier Score: 0.1628, Accuracy: 0.7605
Fold 9 - Brier Score: 0.1653, Accuracy: 0.7544
Fold 10 - Brier Score: 0.1618, Accuracy: 0.7590

Average Brier Score over 10 folds: 0.1633
Average Accuracy over 10 folds: 0.7574


In [10]:
XW = train_dataW[feature_cols]
yW = train_dataW[target_col].values

# Set up K-fold cross validation
n_splits = 10
kfM = KFold(n_splits=n_splits, shuffle=True, random_state=123)

brier_scores = []
accuracy_scores = []

for fold, (train_index, val_index) in enumerate(kfM.split(XW), 1):
    # Create training and validation splits for the fold
    X_train_cv, X_val_cv = XW.iloc[train_index], XW.iloc[val_index]
    y_train_cv, y_val_cv = yW[train_index], yW[val_index]
    
    # Initialize the LightGBM regressor with your fixed parameters
    modelW = lgb.LGBMRegressor(
        n_estimators=500,
        learning_rate=0.01,
        random_state=123,
        verbose=-1
    )
    
    # Train the model
    modelW.fit(X_train_cv, y_train_cv)
    
    # Predict the continuous point margin on the validation fold
    y_val_pred_margin = modelW.predict(X_val_cv)
    
    # Convert predicted margins to win probabilities
    y_val_pred_prob = margin_to_probability(y_val_pred_margin)
    
    # Convert true margins to binary outcomes (win=1 if margin > 0, else 0)
    y_val_true_label = (y_val_cv > 0).astype(int)
    
    # For accuracy, threshold the predicted probabilities at 0.5
    y_val_pred_label = (y_val_pred_prob > 0.5).astype(int)
    
    # Compute Brier score for the fold
    fold_brier = brier_score(y_val_true_label, y_val_pred_prob)
    brier_scores.append(fold_brier)
    
    # Compute accuracy score for the fold
    fold_accuracy = accuracy_score(y_val_true_label, y_val_pred_label)
    accuracy_scores.append(fold_accuracy)
    
    print(f"Fold {fold} - Brier Score: {fold_brier:.4f}, Accuracy: {fold_accuracy:.4f}")

# Compute average scores across folds
avg_brier = np.mean(brier_scores)
avg_accuracy = np.mean(accuracy_scores)
print(f"\nAverage Brier Score over {n_splits} folds: {avg_brier:.4f}")
print(f"Average Accuracy over {n_splits} folds: {avg_accuracy:.4f}")

Fold 1 - Brier Score: 0.1422, Accuracy: 0.7888
Fold 2 - Brier Score: 0.1477, Accuracy: 0.7805
Fold 3 - Brier Score: 0.1505, Accuracy: 0.7760
Fold 4 - Brier Score: 0.1469, Accuracy: 0.7818
Fold 5 - Brier Score: 0.1465, Accuracy: 0.7840
Fold 6 - Brier Score: 0.1429, Accuracy: 0.7901
Fold 7 - Brier Score: 0.1449, Accuracy: 0.7879
Fold 8 - Brier Score: 0.1444, Accuracy: 0.7888
Fold 9 - Brier Score: 0.1446, Accuracy: 0.7860
Fold 10 - Brier Score: 0.1463, Accuracy: 0.7869

Average Brier Score over 10 folds: 0.1457
Average Accuracy over 10 folds: 0.7851


In [11]:
pickle.dump(modelM, open('./model/lgbm_modelM.pkl','wb'))
pickle.dump(modelW, open('./model/lgbm_modelW.pkl','wb'))

In [12]:
team_data25M = df_longM[df_longM['Season']==2025].reset_index(drop=True)
team_data25M.to_csv('./data/team_data.csv', index=False)
team_data25W = df_longW[df_longW['Season']==2025].reset_index(drop=True)

In [13]:
def get_preds(df, model, TeamID_A, TeamID_B):
    feature_cols = ['ORPerc_diff_7','TOPerc_diff_7','DefEff_diff_7','EffFG_diff_7','FTPerc_diff_7']
    data_team_A = df[df['Team']==TeamID_A][feature_cols].tail(1).reset_index(drop=True).add_suffix('_A')
    data_team_B = df[df['Team']==TeamID_B][feature_cols].tail(1).reset_index(drop=True).add_suffix('_B')

    pred_data = pd.concat([data_team_A, data_team_B], axis=1)

    preds_margin = model.predict(pred_data)
    preds_prob = margin_to_probability(preds_margin)
    return preds_prob

In [14]:
submission = pd.read_csv('./data/SampleSubmissionStage2.csv')
display(submission.head())
display(submission.tail())

,ID,Pred
0,2025_1101_1102,0.5
1,2025_1101_1103,0.5
2,2025_1101_1104,0.5
3,2025_1101_1105,0.5
4,2025_1101_1106,0.5


,ID,Pred
131402,2025_3477_3479,0.5
131403,2025_3477_3480,0.5
131404,2025_3478_3479,0.5
131405,2025_3478_3480,0.5
131406,2025_3479_3480,0.5


In [15]:
%%time
print("Getting teams.")
submission_data_xgb = []

c = 0
for row in tqdm.tqdm(submission.itertuples()):
    year = row.ID.split('_')[0]
    teamA_ID = int(row.ID.split('_')[1])
    teamB_ID = int(row.ID.split('_')[2])
    label = year + '_' + str(teamA_ID) + '_' + str(teamB_ID)
    
    try:
        if (teamA_ID < 3000):
            submission_data_xgb.append([label, get_preds(df_longM, modelM, teamA_ID, teamB_ID)[0]])
        else:
            submission_data_xgb.append([label, get_preds(df_longW, modelW, teamA_ID, teamB_ID)[0]])
    except Exception as e:
        print(e)
        c+=1
        submission_data_xgb.append([label, 0.5])
print(c)

Getting teams.


131407it [05:12, 420.27it/s]

0
CPU times: total: 41min 6s
Wall time: 5min 12s


In [16]:
submission_data_xgb_df = pd.DataFrame(submission_data_xgb, columns = ['ID','Pred'])
submission_data_xgb_df.head()

,ID,Pred
0,2025_1101_1102,0.969173
1,2025_1101_1103,0.437453
2,2025_1101_1104,0.488222
3,2025_1101_1105,0.836987
4,2025_1101_1106,0.408826


In [17]:
submission_data_xgb_df.to_csv('./data/complete_submission.csv', index=False)

In [18]:
get_preds(df_longM, modelM, 1401, 1463)

array([0.14222267])